# Phase 1 — Proof Intuition

Empirical companion to `notes/phase1-proof.md`. We numerically demonstrate that

1. $E[X + Y] = E[X] + E[Y]$ holds **regardless of dependence** between $X$ and $Y$, and
2. $E[XY] = E[X]\,E[Y]$ holds **only under independence** (or zero covariance more generally).

Nothing here proves anything — this is a sanity check against the analytic results. The simulation just confirms that the algebra in the notes is not lying.

In [ ]:
import numpy as np
import pandas as pd

RNG = np.random.default_rng(seed=42)
N = 200_000   # sample size — large enough for ~3 decimal digit accuracy

## 1. The headline counterexample: $X = Y$ on $\{-1, +1\}$

Construct the pair $(X, Y)$ from a single fair coin flip recoded to $\{-1, +1\}$. With probability $1/2$ both are $-1$; with probability $1/2$ both are $+1$.

Marginals: $E[X] = E[Y] = 0$.
Sum: $E[X + Y]$ should be $0$ (matches linearity).
Product: $E[XY] = 1$ (since $XY = 1$ always when $X = Y$ and $X^2 = 1$). This contradicts $E[X]\,E[Y] = 0$.

In [ ]:
X = RNG.choice([-1, 1], size=N)
Y = X.copy()  # perfect dependence: Y is identically X

results = {
    "E[X]":          X.mean(),
    "E[Y]":          Y.mean(),
    "E[X+Y]":        (X + Y).mean(),
    "E[X] + E[Y]":   X.mean() + Y.mean(),
    "E[XY]":         (X * Y).mean(),
    "E[X] * E[Y]":   X.mean() * Y.mean(),
}
pd.Series(results).round(4).to_frame("empirical")

Read the table:

- `E[X+Y]` ≈ `E[X] + E[Y]` ≈ `0` — **linearity holds** even though $X$ and $Y$ are perfectly dependent.
- `E[XY]` ≈ `1`, but `E[X] * E[Y]` ≈ `0` — the product formula **fails** by exactly the covariance, which is $1$ here.

Linearity does not care that $X = Y$. The product identity collapses entirely.

## 2. The independent case (for contrast)

Now sample $X$ and $Y$ as independent fair $\pm 1$ coin flips. Both identities should now hold.

In [ ]:
X_ind = RNG.choice([-1, 1], size=N)
Y_ind = RNG.choice([-1, 1], size=N)

results_ind = {
    "E[X]":          X_ind.mean(),
    "E[Y]":          Y_ind.mean(),
    "E[X+Y]":        (X_ind + Y_ind).mean(),
    "E[X] + E[Y]":   X_ind.mean() + Y_ind.mean(),
    "E[XY]":         (X_ind * Y_ind).mean(),
    "E[X] * E[Y]":   X_ind.mean() * Y_ind.mean(),
}
pd.Series(results_ind).round(4).to_frame("empirical")

Now `E[XY]` ≈ `E[X] * E[Y]` ≈ `0`. The product identity is recovered the moment independence holds.

## 3. Sweeping dependence: $E[X + Y]$ stable, $E[XY]$ varies

Generate correlated Gaussian pairs $(X, Y)$ with $E[X] = E[Y] = 0$ and varying correlation $\rho$. Across all values of $\rho$, the sum $E[X + Y]$ stays at $0$. The product $E[XY] = \rho$ tracks the dependence exactly.

In [ ]:
def sample_gaussian_pair(rho: float, size: int, rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    """Draw `size` samples from a bivariate standard normal with correlation rho."""
    Z1 = rng.standard_normal(size)
    Z2 = rng.standard_normal(size)
    X = Z1
    Y = rho * Z1 + np.sqrt(1.0 - rho**2) * Z2
    return X, Y

rhos = [-0.9, -0.5, -0.2, 0.0, 0.2, 0.5, 0.9]
rows = []
for rho in rhos:
    X, Y = sample_gaussian_pair(rho, N, RNG)
    rows.append({
        "rho":        rho,
        "E[X]":       X.mean(),
        "E[Y]":       Y.mean(),
        "E[X+Y]":     (X + Y).mean(),
        "E[XY]":      (X * Y).mean(),
        "E[X]E[Y]":   X.mean() * Y.mean(),
    })
pd.DataFrame(rows).round(4)

The `E[X+Y]` column is statistically indistinguishable from $0$ for every value of $\rho$ — that is the whole content of linearity. The `E[XY]` column moves with $\rho$: it equals $\rho$ in this Gaussian setup (by construction, since $\text{Cov}(X, Y) = \rho$ and the marginals are zero-mean). The gap between `E[XY]` and `E[X]E[Y]` *is* the covariance.

## Takeaway

The mean of a sum is a structurally trivial operation — it only needs marginalization, which is always available. The mean of a product is a structurally non-trivial operation — it needs the joint distribution to factor, which is what independence buys. Phase 2 turns this asymmetry into the variance formula, where it stops being a curiosity and starts costing real money.